In [2]:
import os
import re
import math
import pickle 
from pathlib import Path 

import requests
import pandas as pd
import sqlalchemy as db
import plotly.express as px
import plotly.graph_objects as go
from huggingface_hub import hf_hub_download
from imdbinfo.services import get_movie
from tqdm.auto import tqdm 

In [3]:
if os.path.exists('utils.py'):
    os.remove('utils.py')

raw_url = "https://raw.githubusercontent.com/astaileyyoung/CineFace/research/research/utils.py"

response = requests.get(raw_url)

if response.status_code == 200:
    with open("utils.py", "wb") as f:
        f.write(response.content)
    from utils import *
    print("✅ Success! utils.py is now actual code.")
else:
    print(f"❌ Failed to download. Error code: {response.status_code}")

✅ Success! utils.py is now actual code.


In [4]:
from utils import *

In [5]:
def time_to_seconds(time_str):
    if not time_str:
        return None
    # Match optional hours and optional minutes using regular expressions
    # \d+ looks for one or more digits
    hours_match = re.search(r'(\d+)\s*h', time_str)
    minutes_match = re.search(r'(\d+)\s*m', time_str)
    
    # Extract the numbers if found, otherwise default to 0
    hours = int(hours_match.group(1)) if hours_match else 0
    minutes = int(minutes_match.group(1)) if minutes_match else 0
    
    # Calculate total seconds
    total_seconds = (hours * 3600) + (minutes * 60)
    return total_seconds

In [6]:
layout = {
    'title': {
        'text': '',
        'font': {'size': 22, 'family': 'Raleway', 'color': 'white'},
        'x': 0.5,
        'y': 0.95,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    'xaxis': {
        'title': {
            'text': '',
            'font': {'size': 18, 'family': 'Raleway', 'color': 'white'} 
        },
        'tickfont': {'size': 14, 'family': 'Roboto', 'color': 'white'}
    },
    'yaxis': {
        'layer': 'below traces',
        'title': {
            'text': '',
            'font': {'size': 18, 'family': 'Raleway', 'color': 'white'} 
        },
        'tickfont': {'size': 14, 'family': 'Roboto', 'color': 'white'}
    },
    'font': {'color': 'white'},
    'paper_bgcolor': '#1A1A2E',
    'plot_bgcolor': 'rgba(61, 61, 61, 0)'
}

# Load IMDb Dataset

In [7]:
# username = "amos"
# password = "M0$hicat"
# host = "192.168.0.131"
# port = "3306"
# database = "imdb"
# connection_string = f'mysql+pymysql://{username}:{password}@{host}:{port}/{database}'
# engine = db.create_engine(connection_string)

In [8]:
# query = """
#     SELECT *
#     FROM title_basics tb 
#     WHERE tb.tconst IN (
#         SELECT titleId
#         FROM title_akas
#         WHERE region = 'US'
#         )
#         AND `startYear` >= 1948
#         AND `startYear` <= 1958
#         AND `titleType` = 'movie'
#         AND (genres NOT LIKE '%%documentary%%' OR genres IS NULL) 
# """
# imdb_df = pd.read_sql(query, engine)
# imdb_df

In [9]:
# imdb_df[imdb_df['imdb_id'] == 42788]

In [10]:
# target_studios = {
#     '20th Century Fox',
#     '20th Century-Fox',
#     '20th Century-Fox Film',
#     '20th Century-Fox Film Company',
#     'Twentieth Century Fox',
#     'Twentieth Century Fox Film Company',
#     'Columbia Pictures',
#     'Metro-Goldwyn-Mayer (MGM)',
#     'Paramount Distribution',
#     'Paramount Famous Lasky Corporation',
#     'Paramount Pictures',
#     'RKO Radio Pictures',
#     'RKO Pathé Pictures',
#     'Universal Pictures',
#     'Universal Classics',
#     'United Artists',
#     'Warner Bros.'
# }

In [11]:
# def is_major(distribution_data):
#     is_major = 0
#     for distributor in distribution_data:
#         if distributor.name in target_studios and 'theatrical' in distributor.attributes and ('United States' in distributor.countries or 'US' in distributor.countries or 'USA' in distributor.countries):
#             is_major = 1
#             break 
#     return is_major

In [12]:
# Path("./data/temp").mkdir(exist_ok=True)

# rows = []
# for idx, row in tqdm(imdb_df.iloc[:].iterrows(), total=imdb_df.shape[0]):
#     fp = Path("./data/temp") / f"{row['tconst']}.pkl"
#     if fp.exists():
#         with open(fp, "rb") as f:
#             result = pickle.load(f)
#     else:
#         result = get_movie(row['tconst'])

#     if not 'distribution' in result.company_credits:
#         continue 

#     row['aspect_ratio'] = "|".join([",".join(x) for x in result.aspect_ratios])
#     row['printed_formats'] = "|".join(result.printed_formats)
#     row['colorations'] = '|'.join(result.colorations)
#     row['production_companies'] = '|'.join(result.production)
#     row['countries'] = "|".join(result.countries)
#     row['rating'] = result.rating
#     row['processes'] = "|".join(result.processes)
#     row['distribution_companies'] = "|".join([x.name for x in result.company_credits['distribution']])
#     row['is_major'] = is_major(result.company_credits['distribution'])

#     with open(fp, "wb") as f:
#         pickle.dump(result, f)

#     rows.append(row)
# imdb_meta_df = pd.DataFrame(rows)
# imdb_meta_df

In [13]:
# imdb_meta_df['imdb_id'] = imdb_meta_df['tconst'].map(lambda x: int(x[2:]))
# # imdb_meta_df = imdb_meta_df[imdb_meta_df['countries'].str.contains("United States")]
# imdb_meta_df = imdb_meta_df.rename({'startYear': 'year'}, axis=1)

In [14]:
# # 1. Evaluate every single row without changing the index shape
# # If 'distribution_companies' is NaN (float), we default it to an empty string '' so split() doesn't crash
# mask = imdb_meta_df['distribution_companies'].apply(
#     lambda x: any(studio.strip() in target_studios for studio in x.split('|')) if isinstance(x, str) else False
# )

# # 2. Filter directly using standard boolean indexing (completely bulletproof)
# population = imdb_meta_df[mask]

# # Display sorted results
# population[[
#     'imdb_id', 'primaryTitle', 'year', 'distribution_companies', 'runtimeMinutes'
# ]].sort_values(by='runtimeMinutes', ascending=False)

In [15]:
# imdb_meta_df.to_parquet("./data/widescreen_sample.parquet")

In [16]:
population = pd.read_parquet("./data/widescreen_sample.parquet")
population = population[population['is_major'] == 1]
population

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,imdb_id,aspect_ratio,printed_formats,colorations,production_companies,countries,rating,processes,distribution_companies,is_major
11,tt0039213,movie,Borrowed Trouble,Borrowed Trouble,0,1948,None,58.0,"Drama,Western",39213,"1.37 : 1,",35 mm,Black and White,Hopalong Cassidy Productions Inc.,United States,6.3,Spherical,United Artists|United Artists|United Artists|U...,1
13,tt0039248,movie,El casado casa quiere,El casado casa quiere,0,1948,None,83.0,"Action,Comedy,Drama",39248,"1.37 : 1,",35 mm,Black and White,Ramex Films,Mexico,5.6,Spherical,RKO Radio Pictures de México|Clasa-Mohme|RKO R...,1
20,tt0039304,movie,Daybreak,Daybreak,0,1948,None,75.0,Drama,39304,"1.37 : 1,",35 mm,Black and White,Sydney Box Productions,United Kingdom,6.7,Spherical,General Film Distributors (GFD)|Kommunenes Fil...,1
28,tt0039550,movie,Larceny,Larceny,0,1948,None,89.0,"Crime,Drama,Film-Noir",39550,"1.37 : 1,",35 mm,Black and White,Universal International Pictures (UI),United States,6.8,Spherical,Universal Pictures|Empire Universal Films|Gene...,1
29,tt0039613,movie,Mary Lou,Mary Lou,0,1948,None,65.0,"Music,Romance",39613,"1.37 : 1,",35 mm,Black and White,Sam Katzman Productions,United States,NaN,Spherical,Columbia Pictures|Columbia Pictures of Canada|...,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7280,tt0053935,movie,Ice Cold in Alex,Ice Cold in Alex,0,1958,None,130.0,"Adventure,Drama,War",53935,"1.66 : 1,",35 mm,Black and White|Black and White,Associated British Picture Corporation (ABPC),United Kingdom,7.7,Spherical,Associated British-Pathé|Lehmacher Filmverleih...,1
7282,tt0054192,movie,The Poacher's Daughter,Sally's Irish Rogue,0,1958,None,74.0,Comedy,54192,"1.37 : 1,",35 mm,Black and White,Emmett Dalton Productions,United Kingdom,4.9,Spherical,British Lion Film Corporation|RKO Radio Pictur...,1
7290,tt0122119,movie,Island Women,Island Women,0,1958,None,72.0,Drama,122119,"1.37 : 1,",35 mm,Black and White,Security Pictures,United States,6.8,,United Artists|United Artists,1
7294,tt0135834,movie,Cerf-volant du bout du monde,Cerf-volant du bout du monde,0,1958,None,82.0,"Adventure,Family,Fantasy",135834,"1.37 : 1,",35 mm,Color,Garance Productions|Les Films Jean Tourane|Coc...,France|China,6.8,Spherical,Cocinor|Xerox Films|Paramount Pictures|Tamasa ...,1


In [17]:
population_g = population.groupby("startYear").count()
population_g = population_g.reset_index()
population_g

,startYear,tconst,titleType,primaryTitle,originalTitle,isAdult,endYear,runtimeMinutes,genres,imdb_id,aspect_ratio,printed_formats,colorations,production_companies,countries,rating,processes,distribution_companies,is_major
0,1948,267,267,267,267,267,0,267,267,267,267,267,267,267,267,259,267,267,267
1,1949,262,262,262,262,262,0,262,262,262,262,262,262,262,262,261,262,262,262
2,1950,278,278,278,278,278,0,278,278,278,278,278,278,278,278,276,278,278,278
3,1951,298,298,298,298,298,0,298,298,298,298,298,298,298,298,297,298,298,298
4,1952,282,282,282,282,282,0,282,282,282,282,282,282,282,282,282,282,282,282
5,1953,304,304,304,304,304,0,302,304,304,304,304,304,304,304,292,304,304,304
6,1954,217,217,217,217,217,0,216,216,217,217,217,217,217,217,216,217,217,217
7,1955,216,216,216,216,216,0,216,216,216,216,216,216,216,216,215,216,216,216
8,1956,233,233,233,233,233,0,233,233,233,233,233,233,233,233,233,233,233,233
9,1957,279,279,279,279,279,0,279,279,279,279,279,279,279,279,279,279,279,279


In [18]:
fig = px.bar(population_g, x='startYear', y='tconst')
fig.update_layout(layout)

# Get Existing Films

In [19]:
# dw_local_path = hf_hub_download(
#     repo_id="astaileyyoung/CineFaceDB",
#     filename="CineFaceDW_agg.duckdb",
#     repo_type="dataset",
#     local_dir="."
# )

In [20]:
# engine = db.create_engine("duckdb:///CineFaceDW_agg.duckdb")

In [21]:
username = "amos"
password = "M0$hicat"
host = "192.168.0.131"
port = "3306"
database = "CineFaceDW"
connection_string = f'mysql+pymysql://{username}:{password}@{host}:{port}'
engine = db.create_engine(connection_string)

In [22]:
existing = pd.read_sql("""
    SELECT * 
    FROM CineFaceDW.dimWork 
    WHERE kind = 'movie' 
        AND year >= 1948
        AND year <= 1958
""", engine)
existing

,work_id,imdb_id,series_imdb_id,title,year,season,episode,release_date,kind,rating,metacritic_rating,votes,budget,gross,runtime,plot,is_major,title_localized
0,34272,33996,None,Panhandle,1948,None,None,1948-02-22,movie,6.3,NaN,429,None,None,85,"John Sands, a former Texas marshal turns to ra...",0,Panhandle
1,34812,39188,None,Bill and Coo,1948,None,None,1948-03-28,movie,5.6,NaN,342,None,None,61,The feathered residents of Chirpendale are ter...,0,Bill and Coo
2,34816,39195,None,Blanche Fury,1948,None,None,1948-09-08,movie,6.7,NaN,1259,1500000 USD,None,90,The childless widow of Allan Fury bequeaths th...,0,Blanche Fury
3,34823,39220,None,Brighton Rock,1948,None,None,1951-11-07,movie,7.3,NaN,7516,None,72464 USD,92,"In Brighton in 1935, small-time gang leader Pi...",0,Brighton Rock
4,34836,39304,None,Daybreak,1948,None,None,1949-05-13,movie,6.7,NaN,306,None,None,75,A hangman conceals his true identity when he f...,1,Daybreak
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1795,57978,50964,None,Short Cut to Hell,1957,None,None,1957-09-01,movie,6.1,NaN,535,None,None,89,A hired killer's latest contract goes awry whe...,0,Short Cut to Hell
1796,57979,51087,None,The Tin Star,1957,None,None,1957-11-06,movie,7.3,NaN,7602,None,None,93,A cynical former sheriff turned bounty hunter ...,0,The Tin Star
1797,57980,51649,None,The Geisha Boy,1958,None,None,1958-12-17,movie,6.4,NaN,2330,None,None,98,Gilbert Wooley is a second-rate magician who i...,0,The Geisha Boy
1798,57981,51745,None,Houseboat,1958,None,None,1958-11-19,movie,6.6,NaN,10520,None,74 USD,110,"A widower, his three young children, and an It...",0,Houseboat


In [23]:
minors = existing[existing['is_major'] == 0]
minors

,work_id,imdb_id,series_imdb_id,title,year,season,episode,release_date,kind,rating,metacritic_rating,votes,budget,gross,runtime,plot,is_major,title_localized
0,34272,33996,None,Panhandle,1948,None,None,1948-02-22,movie,6.3,NaN,429,None,None,85,"John Sands, a former Texas marshal turns to ra...",0,Panhandle
1,34812,39188,None,Bill and Coo,1948,None,None,1948-03-28,movie,5.6,NaN,342,None,None,61,The feathered residents of Chirpendale are ter...,0,Bill and Coo
2,34816,39195,None,Blanche Fury,1948,None,None,1948-09-08,movie,6.7,NaN,1259,1500000 USD,None,90,The childless widow of Allan Fury bequeaths th...,0,Blanche Fury
3,34823,39220,None,Brighton Rock,1948,None,None,1951-11-07,movie,7.3,NaN,7516,None,72464 USD,92,"In Brighton in 1935, small-time gang leader Pi...",0,Brighton Rock
6,34902,39679,None,Once Upon a Dream,1949,None,None,1949-02-01,movie,5.9,NaN,114,None,None,87,Chaos ensues when a woman wakes up believing t...,0,Once Upon a Dream
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1795,57978,50964,None,Short Cut to Hell,1957,None,None,1957-09-01,movie,6.1,NaN,535,None,None,89,A hired killer's latest contract goes awry whe...,0,Short Cut to Hell
1796,57979,51087,None,The Tin Star,1957,None,None,1957-11-06,movie,7.3,NaN,7602,None,None,93,A cynical former sheriff turned bounty hunter ...,0,The Tin Star
1797,57980,51649,None,The Geisha Boy,1958,None,None,1958-12-17,movie,6.4,NaN,2330,None,None,98,Gilbert Wooley is a second-rate magician who i...,0,The Geisha Boy
1798,57981,51745,None,Houseboat,1958,None,None,1958-11-19,movie,6.6,NaN,10520,None,74 USD,110,"A widower, his three young children, and an It...",0,Houseboat


In [24]:
sample = existing[existing['is_major'] == 1]
sample.shape[0]

866

In [25]:
sample_g = sample.groupby('year').count()
sample_g = sample_g.reset_index(drop=False)
sample_g

,year,work_id,imdb_id,series_imdb_id,title,season,episode,release_date,kind,rating,metacritic_rating,votes,budget,gross,runtime,plot,is_major,title_localized
0,1948,85,85,0,85,0,0,85,85,85,9,85,27,11,85,85,85,85
1,1949,97,97,0,97,0,0,97,97,97,6,97,31,7,97,97,97,97
2,1950,89,89,0,89,0,0,89,89,89,9,89,23,14,89,89,89,89
3,1951,94,94,0,94,0,0,94,94,94,10,94,27,8,94,94,94,94
4,1952,81,81,0,81,0,0,81,81,81,5,81,17,11,81,81,81,81
5,1953,81,81,0,81,0,0,81,81,81,11,81,34,13,81,81,81,81
6,1954,62,62,0,62,0,0,62,62,62,8,62,25,14,62,62,62,62
7,1955,67,67,0,67,0,0,67,67,67,8,67,20,8,67,67,67,67
8,1956,65,65,0,65,0,0,65,65,65,9,65,20,11,65,65,65,65
9,1957,81,81,0,81,0,0,81,81,81,12,81,27,13,81,81,81,81


In [26]:
fig = px.bar(sample_g, x='year', y='work_id')
fig.update_layout(layout)
fig.show()

# Calculate Sample Size

In [27]:
def calculate_yearly_moe(n, N, confidence_level=0.95):
    if n == 0:
        return float('inf')
    
    # Z-score for 95% confidence
    z = 1.96 
    # Assumed sample proportion (0.5 gives the most conservative/maximum error estimate)
    p = 0.5 
    
    # Standard Error
    se = math.sqrt((p * (1 - p)) / n)
    
    # Finite Population Correction Factor
    fpc = math.sqrt((N - n) / (N - 1)) if N > 1 else 1.0
    
    return z * se * fpc

In [28]:
calculate_yearly_moe(866, 2877)

0.027847019832817545

In [29]:
yearly_data = {}
for x in sample_g['year']:
    try:
        yearly_data[x] = (sample_g[sample_g['year'] == x]['work_id'].iloc[0], population_g[population_g['startYear'] == x]['imdb_id'].iloc[0])
    except Exception as e:
        print(x, e)
yearly_data

{1948: (85, 267),
 1949: (97, 262),
 1950: (89, 278),
 1951: (94, 298),
 1952: (81, 282),
 1953: (81, 304),
 1954: (62, 217),
 1955: (67, 216),
 1956: (65, 233),
 1957: (81, 279),
 1958: (64, 241)}

In [30]:
print(f"{'Year':<6} | {'Population (N)':<15} | {'Sample (n)':<10} | {'Sampling Rate':<15} | {'Margin of Error':<15}")
print("-" * 65)

data = []
for year, (n, N) in yearly_data.items():
    if n > N:
        n = N
    moe = calculate_yearly_moe(n, N)
    rate = (n / N) * 100
    print(f"{year:<6} | {N:<15} | {n:<10} | {rate:>12.1f}% | {moe * 100:>13.2f}%")
    d = {
        "year": year,
        "rate": rate,
        "population": N,
        "sample": n,
        "moe": moe
    }
    data.append(d)
moe_df = pd.DataFrame(data)
moe_df.round(3)

Year   | Population (N)  | Sample (n) | Sampling Rate   | Margin of Error
-----------------------------------------------------------------
1948   | 267             | 85         |         31.8% |          8.79%
1949   | 262             | 97         |         37.0% |          7.91%
1950   | 278             | 89         |         32.0% |          8.58%
1951   | 298             | 94         |         31.5% |          8.38%
1952   | 282             | 81         |         28.7% |          9.21%
1953   | 304             | 81         |         26.6% |          9.34%
1954   | 217             | 62         |         28.6% |         10.54%
1955   | 216             | 67         |         31.0% |          9.97%
1956   | 233             | 65         |         27.9% |         10.34%
1957   | 279             | 81         |         29.0% |          9.19%
1958   | 241             | 64         |         26.6% |         10.52%


,year,rate,population,sample,moe
0,1948,31.835,267,85,0.088
1,1949,37.023,262,97,0.079
2,1950,32.014,278,89,0.086
3,1951,31.544,298,94,0.084
4,1952,28.723,282,81,0.092
5,1953,26.645,304,81,0.093
6,1954,28.571,217,62,0.105
7,1955,31.019,216,67,0.100
8,1956,27.897,233,65,0.103
9,1957,29.032,279,81,0.092


In [31]:
def calculate_required_sample(N, target_moe=0.03):
    z = 1.96  # 95% confidence
    p = 0.5   # Worst-case variance for maximum conservative target
    
    # Baseline sample size for an infinite population
    n_0 = (z**2 * p * (1 - p)) / (target_moe**2)
    
    # Apply Finite Population Correction backwards to find required n for this population
    n = n_0 / (1 + ((n_0 - 1) / N))
    
    return math.ceil(n)

In [32]:
print(f"{'Year':<6} | {'Population (N)':<15} | {'Required Sample (n)':<20} | {'Required Rate':<15}")
print("-" * 65)

for year, (current_n, N) in yearly_data.items():
    # Calculates required sample for a +/-3% margin of error (0.03)
    required_n = calculate_required_sample(N, target_moe=0.03)
    rate = (required_n / N) * 100
    
    print(f"{year:<6} | {N:<15} | {required_n:<20} | {rate:>12.1f}%")

Year   | Population (N)  | Required Sample (n)  | Required Rate  
-----------------------------------------------------------------
1948   | 267             | 214                  |         80.1%
1949   | 262             | 211                  |         80.5%
1950   | 278             | 221                  |         79.5%
1951   | 298             | 234                  |         78.5%
1952   | 282             | 224                  |         79.4%
1953   | 304             | 237                  |         78.0%
1954   | 217             | 181                  |         83.4%
1955   | 216             | 180                  |         83.3%
1956   | 233             | 192                  |         82.4%
1957   | 279             | 222                  |         79.6%
1958   | 241             | 197                  |         81.7%


In [33]:
not_in_population = sample[~sample['imdb_id'].isin(population['imdb_id'])]
not_in_population

,work_id,imdb_id,series_imdb_id,title,year,season,episode,release_date,kind,rating,metacritic_rating,votes,budget,gross,runtime,plot,is_major,title_localized
488,35445,44214,None,White Corridors,1951,None,None,1951-10-01,movie,7.0,NaN,210,None,None,102,"Hospital drama set at the Yeoman's Hospital, i...",1,White Corridors
665,35628,46181,None,Personal Affair,1953,None,None,1954-01-15,movie,6.5,NaN,664,None,None,82,"In a 1950s British village, a teenager, who is...",1,Personal Affair
1102,36083,51492,None,Count Five and Die,1957,None,None,1958-03-01,movie,6.5,NaN,247,None,None,92,American and British counter-espionage combine...,1,Count Five and Die
1346,53778,45681,None,Desperate Moment,1953,None,None,1953-08-01,movie,6.6,NaN,245,None,None,88,Simon Van Halder (Sir Dirk Bogarde) is accused...,1,Desperate Moment


In [34]:
not_in_sample = population[~population['imdb_id'].isin(sample['imdb_id'])]
not_in_sample[['primaryTitle', 'startYear', 'distribution_companies', 'runtimeMinutes']].sort_values(by='runtimeMinutes', ascending=False)

,primaryTitle,startYear,distribution_companies,runtimeMinutes
6458,Quiet Flows the Don,1957,Gala Film Distributors|Deutsche Film Hansa|Tel...,330.0
4200,Gunfighters of the Northwest,1954,Columbia Pictures|Columbia Pictures of Canada|...,315.0
2068,Captain Video: Master of the Stratosphere,1951,Columbia Pictures|Columbia Pictures of Canada|...,287.0
4344,Riding with Buffalo Bill,1954,Columbia Pictures|Columbia Pictures Proprietar...,280.0
131,Congo Bill,1948,Columbia Pictures|Columbia Pictures of Canada|...,270.0
...,...,...,...,...
3973,Winter Hilarities,1953,RKO Radio Pictures,45.0
5284,Rocky Marciano vs. Archie Moore,1955,Theater Network Television|United Artists,19.0
3956,Mr. Magoo Cartoon Merry-Go-Round,1953,Columbia Pictures,NaN
3957,Short Subject Star Parade,1953,Columbia Pictures,NaN


# By Year

In [35]:
query = """
    SELECT fw.*, dw.year
    FROM CineFaceDW.dimWork dw
    INNER JOIN CineFaceDW.factWork fw ON dw.work_id = fw.work_id
    WHERE dw.year >= 1948
        AND dw.year <= 1958
        AND is_major = 1
"""
sample_data = pd.read_sql(query, engine)
sample_data

,fact_id,imdb_id,work_id,avg_size,avg_size_id,avg_size_t1,avg_size_t1_id,max_size,max_size_id,min_size,...,z_v_density_tv_g,z_pct_face_tv,z_pct_face_tv_g,z_v_pct_face_tv,z_v_pct_face_tv_g,z_pct_top1_tv,z_pct_top1_tv_g,z_v_pct_top1_tv,z_v_pct_top1_tv_g,year
0,2470,39304,34836,0.0279,0.0636,0.0347,0.0636,0.2347,0.1444,0.0005,...,None,None,None,None,None,None,None,None,None,1948
1,2924,39550,34887,0.0313,0.0618,0.0414,0.0669,0.2702,0.2685,0.0003,...,None,None,None,None,None,None,None,None,None,1948
2,2997,40002,34945,0.0257,0.0620,0.0353,0.0620,0.2157,0.2157,0.0005,...,None,None,None,None,None,None,None,None,None,1948
3,2999,40064,34947,0.0108,0.0329,0.0148,0.0334,0.1681,0.1443,0.0003,...,None,None,None,None,None,None,None,None,None,1948
4,3000,40087,34948,0.0315,0.0649,0.0418,0.0650,0.2626,0.2600,0.0006,...,None,None,None,None,None,None,None,None,None,1948
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
863,4735,52228,53839,0.0258,0.1035,0.0339,0.1037,0.3254,0.2314,0.0009,...,None,None,None,None,None,None,None,None,None,1958
864,3139,41623,55338,0.0177,0.0501,0.0219,0.0501,0.1296,0.0870,0.0016,...,None,None,None,None,None,None,None,None,None,1949
865,8222,169625,55409,0.0324,0.0879,0.0387,0.0879,0.3009,0.0879,0.0016,...,None,None,None,None,None,None,None,None,None,1952
866,3820,45831,55959,0.0199,NaN,0.0220,NaN,0.0545,NaN,0.0009,...,None,None,None,None,None,None,None,None,None,1953


In [36]:
sample_data_g = sample_data.groupby('year').mean()
sample_data_g = sample_data_g.reset_index()
sample_data_g

,year,fact_id,imdb_id,work_id,avg_size,avg_size_id,avg_size_t1,avg_size_t1_id,max_size,max_size_id,...,z_v_density_tv,z_v_density_tv_g,z_pct_face_tv,z_pct_face_tv_g,z_v_pct_face_tv,z_v_pct_face_tv_g,z_pct_top1_tv,z_pct_top1_tv_g,z_v_pct_top1_tv,z_v_pct_top1_tv_g
0,1948,3199.000000,42143.070588,35931.341176,0.022095,0.060556,0.030044,0.061802,0.267716,0.253018,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1949,3159.731959,41598.360825,37234.278351,0.022271,0.061292,0.030048,0.062291,0.264586,0.239054,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1950,3448.404494,44178.449438,36933.797753,0.023231,0.065842,0.031973,0.067154,0.281758,0.257030,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1951,3597.627660,44572.989362,36167.478723,0.022355,0.062218,0.030248,0.063172,0.254330,0.227949,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1952,3755.951220,46373.512195,37244.719512,0.022204,0.065179,0.030604,0.066082,0.273910,0.238100,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1953,3850.780488,46039.902439,37170.000000,0.021245,0.066169,0.029965,0.067521,0.272044,0.244806,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1954,3987.693548,47155.016129,36294.322581,0.019413,0.056745,0.027039,0.057640,0.237356,0.215389,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1955,4172.925373,48335.895522,38095.343284,0.017896,0.052682,0.025187,0.053666,0.201499,0.173566,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1956,4409.707692,49487.738462,37699.092308,0.020891,0.060509,0.029754,0.061480,0.233902,0.206205,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1957,4486.111111,51554.308642,38876.432099,0.021183,0.059935,0.029606,0.061022,0.236284,0.196777,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
sample_data_g = sample_data_g.merge(
    moe_df,
    how='inner',
    on='year'
)
sample_data_g

,year,fact_id,imdb_id,work_id,avg_size,avg_size_id,avg_size_t1,avg_size_t1_id,max_size,max_size_id,...,z_v_pct_face_tv,z_v_pct_face_tv_g,z_pct_top1_tv,z_pct_top1_tv_g,z_v_pct_top1_tv,z_v_pct_top1_tv_g,rate,population,sample,moe
0,1948,3199.000000,42143.070588,35931.341176,0.022095,0.060556,0.030044,0.061802,0.267716,0.253018,...,NaN,NaN,NaN,NaN,NaN,NaN,31.835206,267,85,0.087925
1,1949,3159.731959,41598.360825,37234.278351,0.022271,0.061292,0.030048,0.062291,0.264586,0.239054,...,NaN,NaN,NaN,NaN,NaN,NaN,37.022901,262,97,0.079116
2,1950,3448.404494,44178.449438,36933.797753,0.023231,0.065842,0.031973,0.067154,0.281758,0.257030,...,NaN,NaN,NaN,NaN,NaN,NaN,32.014388,278,89,0.085807
3,1951,3597.627660,44572.989362,36167.478723,0.022355,0.062218,0.030248,0.063172,0.254330,0.227949,...,NaN,NaN,NaN,NaN,NaN,NaN,31.543624,298,94,0.083772
4,1952,3755.951220,46373.512195,37244.719512,0.022204,0.065179,0.030604,0.066082,0.273910,0.238100,...,NaN,NaN,NaN,NaN,NaN,NaN,28.723404,282,81,0.092093
5,1953,3850.780488,46039.902439,37170.000000,0.021245,0.066169,0.029965,0.067521,0.272044,0.244806,...,NaN,NaN,NaN,NaN,NaN,NaN,26.644737,304,81,0.093415
6,1954,3987.693548,47155.016129,36294.322581,0.019413,0.056745,0.027039,0.057640,0.237356,0.215389,...,NaN,NaN,NaN,NaN,NaN,NaN,28.571429,217,62,0.105431
7,1955,4172.925373,48335.895522,38095.343284,0.017896,0.052682,0.025187,0.053666,0.201499,0.173566,...,NaN,NaN,NaN,NaN,NaN,NaN,31.018519,216,67,0.099670
8,1956,4409.707692,49487.738462,37699.092308,0.020891,0.060509,0.029754,0.061480,0.233902,0.206205,...,NaN,NaN,NaN,NaN,NaN,NaN,27.896996,233,65,0.103438
9,1957,4486.111111,51554.308642,38876.432099,0.021183,0.059935,0.029606,0.061022,0.236284,0.196777,...,NaN,NaN,NaN,NaN,NaN,NaN,29.032258,279,81,0.091895


## Avg. Size

In [38]:
x = sample_data_g['year'].tolist()
y = sample_data_g['avg_size'].round(3)

error = y * sample_data_g['moe']
y_upper = (y + error).tolist()
y_lower = (y - error).tolist()

fig = go.Figure([
    go.Scatter(
        x=x + x[::-1],
        y=y_upper + y_lower[::-1],
        fill='toself',
        line=dict(color='#FF6B6B'),
fillcolor='rgba(255, 107, 107, 0.5)',
        hoverinfo='skip',
        showlegend=False
    ),
    go.Scatter(
        x=x, y=y.tolist(),
        line=dict(color='rgb(255, 255, 255)'),
        name='measurement'
    )
])
fig.update_layout(layout)
fig.update_layout(dict(title=dict(text="Avg. Face Size, 1948-1958", y=0.9, x=0.5)))
fig.show()

## Density

In [39]:
x = sample_data_g['year'].tolist()
y = sample_data_g['avg_f_per_fr'].round(3)

error = y * sample_data_g['moe']
y_upper = (y + error).tolist()
y_lower = (y - error).tolist()

fig = go.Figure([
    go.Scatter(
        x=x + x[::-1],
        y=y_upper + y_lower[::-1],
        fill='toself',
        line=dict(color='#FF6B6B'),
fillcolor='rgba(255, 107, 107, 0.5)',
        hoverinfo='skip',
        showlegend=False
    ),
    go.Scatter(
        x=x, y=y.tolist(),
        line=dict(color='rgb(255, 255, 255)'),
        name='measurement'
    )
])
fig.update_layout(layout)
fig.update_layout(dict(title=dict(text="Avg. Faces per Frame, 1948-1958", y=0.9, x=0.5)))
fig.show()

## Distance

In [40]:
x = sample_data_g['year'].tolist()
y = sample_data_g['avg_dist'].round(3)

error = y * sample_data_g['moe']
y_upper = (y + error).tolist()
y_lower = (y - error).tolist()

fig = go.Figure([
    go.Scatter(
        x=x + x[::-1],
        y=y_upper + y_lower[::-1],
        fill='toself',
        line=dict(color='#FF6B6B'),
fillcolor='rgba(255, 107, 107, 0.5)',
        hoverinfo='skip',
        showlegend=False
    ),
    go.Scatter(
        x=x, y=y.tolist(),
        line=dict(color='rgb(255, 255, 255)'),
        name='measurement'
    )
])
fig.update_layout(layout)
fig.update_layout(dict(title=dict(text="Avg. Distance from Center, 1948-1958", y=0.9, x=0.5)))
fig.show()

## Gini

In [41]:
x = sample_data_g['year'].tolist()
y = sample_data_g['gini'].round(3)

error = y * sample_data_g['moe']
y_upper = (y + error).tolist()
y_lower = (y - error).tolist()

fig = go.Figure([
    go.Scatter(
        x=x + x[::-1],
        y=y_upper + y_lower[::-1],
        fill='toself',
        line=dict(color='#FF6B6B'),
fillcolor='rgba(255, 107, 107, 0.5)',
        hoverinfo='skip',
        showlegend=False
    ),
    go.Scatter(
        x=x, y=y.tolist(),
        line=dict(color='rgb(255, 255, 255)'),
        name='measurement'
    )
])
fig.update_layout(layout)
fig.update_layout(dict(title=dict(text="Gini Score, 1948-1958", y=0.9, x=0.5)))
fig.show()

## Dispersion

In [42]:
x = sample_data_g['year'].tolist()
y = sample_data_g['avg_disp'].round(3)

error = y * sample_data_g['moe']
y_upper = (y + error).tolist()
y_lower = (y - error).tolist()

fig = go.Figure([
    go.Scatter(
        x=x + x[::-1],
        y=y_upper + y_lower[::-1],
        fill='toself',
        line=dict(color='#FF6B6B'),
fillcolor='rgba(255, 107, 107, 0.5)',
        hoverinfo='skip',
        showlegend=False
    ),
    go.Scatter(
        x=x, y=y.tolist(),
        line=dict(color='rgb(255, 255, 255)'),
        name='measurement'
    )
])
fig.update_layout(layout)
fig.update_layout(dict(title=dict(text="Avg. Dispersion, 1948-1958", y=0.9, x=0.5)))
fig.show()

## Horizontal Spread

In [43]:
x = sample_data_g['year'].tolist()
y = sample_data_g['avg_h_spread'].round(3)

error = y * sample_data_g['moe']
y_upper = (y + error).tolist()
y_lower = (y - error).tolist()

fig = go.Figure([
    go.Scatter(
        x=x + x[::-1],
        y=y_upper + y_lower[::-1],
        fill='toself',
        line=dict(color='#FF6B6B'),
fillcolor='rgba(255, 107, 107, 0.5)',
        hoverinfo='skip',
        showlegend=False
    ),
    go.Scatter(
        x=x, y=y.tolist(),
        line=dict(color='rgb(255, 255, 255)'),
        name='measurement'
    )
])
fig.update_layout(layout)
fig.update_layout(dict(title=dict(text="Avg. Horizontal Spread, 1948-1958", y=0.9, x=0.5)))
fig.show()

# Aspect Ratio

In [94]:
query = """
    WITH multiples AS (
        SELECT 
            tar.tconst,
            COUNT(*)
        FROM imdb.aspect_ratio ar 
        INNER JOIN imdb.title_aspect_ratio tar ON ar.id = tar.aspect_ratio_id
        GROUP BY tar.tconst
        HAVING COUNT(*) > 1
    )
    SELECT DISTINCT specification
    FROM imdb.aspect_ratio ar
    INNER JOIN imdb.title_aspect_ratio tar ON ar.id = tar.aspect_ratio_id
    WHERE tar.tconst IN (
        SELECT tconst
        FROM multiples
    )
"""
pop_aspect = pd.read_sql(query, engine)
pop_aspect['specification'].tolist()

['original release',
 '1957 RKO-Scope re-release',
 '',
 'DVD-R version',
 'DVD',
 'USA DVD release',
 'Turner Classic Movies June 2015 airing',
 'negative ratio',
 'intended ratio',
 'theatrical ratio',
 'single-strip 3-D version',
 'dual-strip 3-D version',
 'matted',
 '1969 UK re-release',
 'cropped ratio',
 'CinemaScope',
 'alternate spherical version',
 'shooting ratio',
 'DVD version',
 'Panoramic Screen',
 'cropped wide screen ratio',
 'original aspect ratio',
 'cropped',
 'intended ratio for widescreen theatres',
 'DVD aspect ratio',
 'DVD release',
 'master ratio',
 'CinemaScope version',
 'flat version',
 '3-D version',
 'Pola-Lite 3D theatrical ratio',
 '3D theatrical release & Blu-ray edition',
 'negative ratio & 2D theatrical ratio',
 'alternative 3D theatrical ratio',
 'negative & theatrical ratio',
 'originally intended theatrical ratio',
 'original ratio',
 'spherical version',
 '70 mm prints',
 'intended & theatrical ratio',
 'anamorphic prints',
 'Blu-ray release',
 '

In [45]:
query = """
    SELECT
        fw.*,
        dw.title,
        dw.year,
        dap.ratio,
        dap.ratio_decimal,
        dap.ratio_standardized,
        dap.is_wide,
        dap.is_flat
    FROM CineFaceDW.factWork fw
    INNER JOIN CineFaceDW.dimWork dw ON fw.work_id = dw.work_id
    INNER JOIN CineFaceDW.bridgeAspectRatio bap ON dw.work_id = bap.work_id
    INNER JOIN CineFaceDW.dimAspectRatio dap ON bap.aspect_ratio_id = dap.aspect_ratio_id
    WHERE dw.year >= 1948
        AND dw.year <= 1958
"""
aspect = pd.read_sql(query, engine)
aspect

,fact_id,imdb_id,work_id,avg_size,avg_size_id,avg_size_t1,avg_size_t1_id,max_size,max_size_id,min_size,...,z_pct_top1_tv_g,z_v_pct_top1_tv,z_v_pct_top1_tv_g,title,year,ratio,ratio_decimal,ratio_standardized,is_wide,is_flat
0,2336,33996,34272,0.0149,0.0359,0.0184,0.0359,0.0648,0.0640,0.0006,...,NaN,NaN,NaN,Panhandle,1948,1.37 : 1,1.37,1.33,0,1
1,2444,39188,34812,0.0433,0.0797,0.0531,0.0797,0.1806,0.1177,0.0014,...,NaN,NaN,NaN,Bill and Coo,1948,1.37 : 1,1.37,1.33,0,1
2,2448,39195,34816,0.0234,0.0905,0.0347,0.0907,0.2991,0.2991,0.0002,...,NaN,NaN,NaN,Blanche Fury,1948,1.37 : 1,1.37,1.33,0,1
3,2456,39220,34823,0.0300,0.0668,0.0441,0.0696,0.4144,0.3871,0.0002,...,NaN,NaN,NaN,Brighton Rock,1948,1.37 : 1,1.37,1.33,0,1
4,2470,39304,34836,0.0279,0.0636,0.0347,0.0636,0.2347,0.1444,0.0005,...,NaN,NaN,NaN,Daybreak,1948,1.37 : 1,1.37,1.33,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2173,4660,50964,57978,0.0172,0.0400,0.0233,0.0405,0.2052,0.1393,0.0003,...,NaN,NaN,NaN,Short Cut to Hell,1957,1.85 : 1,1.85,1.85,1,0
2174,4684,51087,57979,0.0188,0.0543,0.0299,0.0555,0.2799,0.2799,0.0003,...,NaN,NaN,NaN,The Tin Star,1957,1.85 : 1,1.85,1.85,1,0
2175,4553,51649,57980,0.0132,0.0291,0.0187,0.0311,0.2489,0.2489,0.0002,...,NaN,NaN,NaN,The Geisha Boy,1958,1.85 : 1,1.85,1.85,1,0
2176,4570,51745,57981,0.0093,0.0308,0.0149,0.0318,0.2077,0.1522,0.0002,...,NaN,NaN,NaN,Houseboat,1958,1.85 : 1,1.85,1.85,1,0


In [46]:
g_cnt = aspect.groupby('ratio_standardized').count()
g_cnt = g_cnt.reset_index()
g_cnt['ratio_standardized'] = g_cnt['ratio_standardized'].astype(str)
g_cnt

,ratio_standardized,fact_id,imdb_id,work_id,avg_size,avg_size_id,avg_size_t1,avg_size_t1_id,max_size,max_size_id,...,z_pct_top1_tv,z_pct_top1_tv_g,z_v_pct_top1_tv,z_v_pct_top1_tv_g,title,year,ratio,ratio_decimal,is_wide,is_flat
0,1.33,1371,1371,1371,1371,1363,1371,1363,1371,1363,...,181,181,181,181,1371,1371,1371,1371,1371,1371
1,1.66,146,146,146,146,146,146,146,146,146,...,0,0,0,0,146,146,146,146,146,146
2,1.85,349,349,349,349,347,349,347,349,347,...,0,0,0,0,349,349,349,349,349,349
3,2.35,312,312,312,312,308,312,308,312,308,...,0,0,0,0,312,312,312,312,312,312


## Sample by Aspect Ratio

In [47]:
fig = px.bar(g_cnt, x='ratio_standardized', y='work_id', color_discrete_sequence=["#3bbdb6"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Count by Aspect Ratio"))
fig.show()

In [48]:
g = aspect[[
    'ratio_standardized',
    'avg_size',
    'avg_f_per_fr',
    'std_f_per_fr',
    'avg_dist',
    'std_dist',
    'gini',
    'avg_disp',
    'avg_h_spread',
    'std_h_spread',
    'pct_tl',
    'pct_tc',
    'pct_tr',
    'pct_ml',
    'pct_mc',
    'pct_mr',
    'pct_bl',
    'pct_bc',
    'pct_br'
]].groupby('ratio_standardized').mean().round(3)
g = g.reset_index()
g['ratio_standardized'] = g['ratio_standardized'].astype(str)
g

,ratio_standardized,avg_size,avg_f_per_fr,std_f_per_fr,avg_dist,std_dist,gini,avg_disp,avg_h_spread,std_h_spread,pct_tl,pct_tc,pct_tr,pct_ml,pct_mc,pct_mr,pct_bl,pct_bc,pct_br
0,1.33,0.021,1.990,1.325,0.368,0.142,0.500,0.203,0.128,0.127,0.116,0.288,0.113,0.108,0.252,0.108,0.005,0.006,0.005
1,1.66,0.021,2.053,1.536,0.354,0.146,0.494,0.206,0.126,0.127,0.100,0.243,0.098,0.121,0.296,0.121,0.007,0.007,0.007
2,1.85,0.020,2.211,1.630,0.360,0.149,0.489,0.211,0.136,0.128,0.102,0.235,0.099,0.125,0.289,0.128,0.007,0.008,0.007
3,2.35,0.014,2.454,2.028,0.373,0.154,0.477,0.221,0.146,0.129,0.106,0.225,0.099,0.140,0.272,0.132,0.009,0.009,0.009


## Size

In [49]:
fig = px.bar(g, x='ratio_standardized', y='avg_size', color_discrete_sequence=["#3bbdb6"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Avg. Size by Aspect Ratio"))
fig.show()

## Density

In [50]:
fig = px.bar(g, x='ratio_standardized', y=['avg_f_per_fr', 'std_f_per_fr'], barmode='group', color_discrete_sequence=["#3bbdb6", "#e97bec"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Avg. Faces per Frame by Aspect Ratio"))
fig.show()

## Distance

In [51]:
fig = px.bar(g, 
             x='ratio_standardized', 
             y=['avg_dist', 'std_dist'], 
             barmode='group', 
             color_discrete_sequence=["#3bbdb6", "#e97bec"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Avg. Distance from Center by Aspect Ratio"))
fig.show()

## Gini

In [52]:
fig = px.bar(g, x='ratio_standardized', y='gini', color_discrete_sequence=["#3bbdb6"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Gini Score by Aspect Ratio"))
fig.show()

## Dispersion

In [53]:
fig = px.bar(g, x='ratio_standardized', y='avg_disp', color_discrete_sequence=["#3bbdb6"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Avg. Dispersion by Aspect Ratio"))
fig.show()

## Horizontal Spread

In [54]:
fig = px.bar(g, 
             x='ratio_standardized', 
             y=['std_h_spread', 'avg_h_spread'], 
             color_discrete_sequence=["#3bbdb6", "#e97bec"],
             barmode='group')
fig.update_layout(layout)
fig.update_layout(title=dict(text="Horizontal Spread by Aspect Ratio"))
fig.show()

## Grid

In [55]:
grid = g[[
    'ratio_standardized',
    'pct_tl',
    'pct_tc',
    'pct_tr',
    'pct_ml',
    'pct_mc',
    'pct_mr',
    'pct_bl',
    'pct_bc',
    'pct_br'
]]
grid
# grid = grid.values.reshape(3, 3)
# grid_norm = grid / grid.sum()
# (grid_norm * 100).round(1)

,ratio_standardized,pct_tl,pct_tc,pct_tr,pct_ml,pct_mc,pct_mr,pct_bl,pct_bc,pct_br
0,1.33,0.116,0.288,0.113,0.108,0.252,0.108,0.005,0.006,0.005
1,1.66,0.100,0.243,0.098,0.121,0.296,0.121,0.007,0.007,0.007
2,1.85,0.102,0.235,0.099,0.125,0.289,0.128,0.007,0.008,0.007
3,2.35,0.106,0.225,0.099,0.140,0.272,0.132,0.009,0.009,0.009


### 1.33

In [56]:
flat = grid[grid['ratio_standardized'] == '1.33'].drop('ratio_standardized', axis=1)
flat_grid = flat.values.reshape(3, 3)
flat_grid_norm = flat_grid / flat_grid.sum()
(flat_grid_norm * 100).round(1)

array([[11.6, 28.8, 11.3],
       [10.8, 25.2, 10.8],
       [ 0.5,  0.6,  0.5]])

In [57]:
plot_grid((flat_grid_norm * 100).round(1), layout=layout)

### 1.85

In [58]:
wide = grid[grid['ratio_standardized'] == '1.85'].drop('ratio_standardized', axis=1)
wide_grid = wide.values.reshape(3, 3)
wide_grid_norm = wide_grid / wide_grid.sum()

In [59]:
plot_grid((wide_grid_norm * 100).round(1), layout=layout)

### 2.35

In [60]:
wide = grid[grid['ratio_standardized'] == '2.35'].drop('ratio_standardized', axis=1)
wide_grid = wide.values.reshape(3, 3)
wide_grid_norm = wide_grid / wide_grid.sum()

In [61]:
plot_grid((wide_grid_norm * 100).round(1), layout=layout)

# Processes

In [62]:
username = "amos"
password = "M0$hicat"
host = "192.168.0.131"
port = "3306"
database = "CineFaceDW"
connection_string = f'mysql+pymysql://{username}:{password}@{host}:{port}/{database}'
engine = db.create_engine(connection_string)

In [63]:
query = """
    SELECT 
        fw.*,
        dw.title,
        dw.year,
        dp.process_name
    FROM CineFaceDW.factWork fw
    INNER JOIN CineFaceDW.dimWork dw ON fw.work_id = dw.work_id
    INNER JOIN CineFaceDW.bridgeProcess as bp ON dw.work_id = bp.work_id
    INNER JOIN CineFaceDW.dimProcess AS dp ON bp.process_id = dp.process_id
    WHERE dw.year >= 1948
        AND dw.year <= 1958
        AND dp.process_name IN ('CinemaScope', 'Todd-AO', 'VistaVision', 'Spherical')
"""
process_df = pd.read_sql(query, engine).round(3)
process_df

,fact_id,imdb_id,work_id,avg_size,avg_size_id,avg_size_t1,avg_size_t1_id,max_size,max_size_id,min_size,...,z_pct_face_tv_g,z_v_pct_face_tv,z_v_pct_face_tv_g,z_pct_top1_tv,z_pct_top1_tv_g,z_v_pct_top1_tv,z_v_pct_top1_tv_g,title,year,process_name
0,2336,33996,34272,0.015,0.036,0.018,0.036,0.065,0.064,0.001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Panhandle,1948,Spherical
1,2444,39188,34812,0.043,0.080,0.053,0.080,0.181,0.118,0.001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Bill and Coo,1948,Spherical
2,2448,39195,34816,0.023,0.090,0.035,0.091,0.299,0.299,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Blanche Fury,1948,Spherical
3,2456,39220,34823,0.030,0.067,0.044,0.070,0.414,0.387,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Brighton Rock,1948,Spherical
4,2470,39304,34836,0.028,0.064,0.035,0.064,0.235,0.144,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Daybreak,1948,Spherical
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1794,4724,52126,57635,0.007,0.016,0.011,0.016,0.073,0.073,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,The Reluctant Debutante,1958,CinemaScope
1795,4845,53236,57637,0.015,0.053,0.024,0.055,0.279,0.279,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,The Saga of Hemp Brown,1958,CinemaScope
1796,4446,49578,57970,0.012,0.034,0.021,0.036,0.087,0.083,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,The Opposite Sex,1956,CinemaScope
1797,4587,51839,57982,0.018,0.073,0.022,0.073,0.116,0.116,0.001,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,The Lady Takes a Flyer,1957,CinemaScope


In [64]:
g = process_df[[
    'process_name',
    'avg_size',
    'std_size',
    'avg_f_per_fr',
    'std_f_per_fr',
    'avg_dist',
    'std_dist',
    'gini',
    'avg_disp',
    'std_disp',
    'avg_h_spread',
    'std_h_spread',
    'avg_v_disc',
    'std_v_disc',
    'pct_tl',
    'pct_tc',
    'pct_tr',
    'pct_ml',
    'pct_mc',
    'pct_mr',
    'pct_bl',
    'pct_bc',
    'pct_br'
]].groupby('process_name').mean()
g = g.reset_index()
g = g[g['process_name'].isin(["CinemaScope", "VistaVision", "Todd-AO", "Spherical"])].round(4)
g

,process_name,avg_size,std_size,avg_f_per_fr,std_f_per_fr,avg_dist,std_dist,gini,avg_disp,std_disp,...,std_v_disc,pct_tl,pct_tc,pct_tr,pct_ml,pct_mc,pct_mr,pct_bl,pct_bc,pct_br
0,CinemaScope,0.0136,0.0147,2.4572,2.0432,0.3727,0.1533,0.4800,0.2204,0.1197,...,0.0717,0.1057,0.2267,0.0982,0.1395,0.2724,0.1317,0.0087,0.0085,0.0087
1,Spherical,0.0212,0.0250,2.0046,1.3464,0.3669,0.1426,0.4975,0.2042,0.1080,...,0.0692,0.1142,0.2817,0.1121,0.1098,0.2560,0.1095,0.0052,0.0061,0.0054
2,Todd-AO,0.0070,0.0090,4.1510,4.8910,0.4080,0.1720,0.4380,0.2380,0.1330,...,0.0640,0.1080,0.2370,0.1280,0.1800,0.2270,0.1010,0.0120,0.0040,0.0020
3,VistaVision,0.0138,0.0159,2.4273,2.0254,0.3760,0.1495,0.4874,0.2107,0.1177,...,0.0678,0.1168,0.2799,0.1126,0.1106,0.2480,0.1142,0.0059,0.0057,0.0062


In [65]:
query = """
    SELECT  
        tb.*,
        p.name
    FROM imdb.title_basics tb
    INNER JOIN imdb.title_process tp ON tb.tconst = tp.tconst
    INNER JOIN imdb.process p ON p.id = tp.process_id
    WHERE p.name IN ('CinemaScope', 'Spherical', 'VistaVision', 'Todd-AO')
        AND startYear >= 1948
        AND startYear <= 1958
"""

population_process_df = pd.read_sql(query, engine)
population_process_df

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,imdb_id,is_major,name
0,tt0021617,movie,Arizona Territory,Arizona Territory,0,1950,None,56.0,"Drama,Western",21617,NaN,Spherical
1,tt0031603,movie,Made in Germany - Die dramatische Geschichte d...,Made in Germany,0,1957,None,97.0,"Biography,Drama",31603,NaN,Spherical
2,tt0033996,movie,Panhandle,Panhandle,0,1948,None,85.0,Western,33996,0.0,Spherical
3,tt0035933,movie,Elephant Fury,Gesprengte Gitter,0,1953,None,83.0,"Drama,War",35933,NaN,Spherical
4,tt0036493,movie,Mystery of the Black Jungle,I misteri della giungla nera,0,1954,None,80.0,"Action,Adventure,Mystery",36493,NaN,Spherical
...,...,...,...,...,...,...,...,...,...,...,...,...
5010,tt39050593,movie,Spring Frolics,Spring Frolics,0,1953,None,45.0,"Animation,Comedy",39050593,NaN,Spherical
5011,tt39050594,movie,Drive-In Frivolities,Drive-In Frivolities,0,1953,None,45.0,"Animation,Comedy",39050594,NaN,Spherical
5012,tt39050595,movie,Drive-In Capers,Drive-In Capers,0,1953,None,45.0,"Animation,Comedy",39050595,NaN,Spherical
5013,tt39050596,movie,April Fool's Frolic,April Fool's Frolic,0,1953,None,45.0,"Animation,Comedy",39050596,NaN,Spherical


In [66]:
pop_g = population_process_df.groupby('name').count()
pop_g

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,imdb_id,is_major
name,,,,,,,,,,,
CinemaScope,374,374,374,374,374,374,0,372,372,374,270
Spherical,4555,4555,4555,4555,4555,4555,0,4547,4532,4555,1245
Todd-AO,3,3,3,3,3,3,0,3,3,3,1
VistaVision,83,83,83,83,83,83,0,83,83,83,63


In [67]:
def calculate_yearly_moe(n, N, confidence_level=0.95):
    if n == 0:
        return float('inf')
    
    # Z-score for 95% confidence
    z = 1.96 
    # Assumed sample proportion (0.5 gives the most conservative/maximum error estimate)
    p = 0.5 
    
    # Standard Error
    se = math.sqrt((p * (1 - p)) / n)
    
    # Finite Population Correction Factor
    fpc = math.sqrt((N - n) / (N - 1)) if N > 1 else 1.0
    
    return z * se * fpc

## Counts

In [68]:
cnts = process_df[['process_name', 'avg_size']].groupby('process_name').count()
cnts = cnts[cnts.index.isin(["CinemaScope", "VistaVision", "Cinerama", "Panavision", "Todd-AO", "Spherical"])]
cnts = cnts.rename({"avg_size": "count"}, axis=1)
cnts

,count
process_name,
CinemaScope,271
Spherical,1464
Todd-AO,1
VistaVision,63


In [69]:
cnts['pop'] = pop_g['tconst']
cnts['moe'] = cnts.apply(lambda x: calculate_yearly_moe(x['count'], x['pop']), axis=1)
cnts

,count,pop,moe
process_name,,,
CinemaScope,271,374,0.031283
Spherical,1464,4555,0.021101
Todd-AO,1,3,0.980000
VistaVision,63,83,0.060977


## Size

In [70]:
g_error = g.merge(cnts,
            how='inner',
            on='process_name')
g_error['avg_size_error'] = g_error['avg_size'] * g_error['moe']
g_error

,process_name,avg_size,std_size,avg_f_per_fr,std_f_per_fr,avg_dist,std_dist,gini,avg_disp,std_disp,...,pct_ml,pct_mc,pct_mr,pct_bl,pct_bc,pct_br,count,pop,moe,avg_size_error
0,CinemaScope,0.0136,0.0147,2.4572,2.0432,0.3727,0.1533,0.4800,0.2204,0.1197,...,0.1395,0.2724,0.1317,0.0087,0.0085,0.0087,271,374,0.031283,0.000425
1,Spherical,0.0212,0.0250,2.0046,1.3464,0.3669,0.1426,0.4975,0.2042,0.1080,...,0.1098,0.2560,0.1095,0.0052,0.0061,0.0054,1464,4555,0.021101,0.000447
2,Todd-AO,0.0070,0.0090,4.1510,4.8910,0.4080,0.1720,0.4380,0.2380,0.1330,...,0.1800,0.2270,0.1010,0.0120,0.0040,0.0020,1,3,0.980000,0.006860
3,VistaVision,0.0138,0.0159,2.4273,2.0254,0.3760,0.1495,0.4874,0.2107,0.1177,...,0.1106,0.2480,0.1142,0.0059,0.0057,0.0062,63,83,0.060977,0.000841


In [71]:
fig = px.bar(g, 
             x='process_name', 
             y=['avg_size', 'std_size'], 
             error_y=g_error['avg_size_error'],
             barmode='group', 
             color_discrete_sequence=["#3bbdb6", "#e97bec"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Avg. Size by Production Process"))
fig.update_traces(error_y=dict(
                      thickness=2,
                      width=6,
                      color="white"
                  ))
fig.show()

## Density

In [72]:
g_error = g.merge(cnts,
            how='inner',
            on='process_name')
g_error['avg_f_per_fr_error'] = g_error['avg_f_per_fr'] * g_error['moe']
g_error

,process_name,avg_size,std_size,avg_f_per_fr,std_f_per_fr,avg_dist,std_dist,gini,avg_disp,std_disp,...,pct_ml,pct_mc,pct_mr,pct_bl,pct_bc,pct_br,count,pop,moe,avg_f_per_fr_error
0,CinemaScope,0.0136,0.0147,2.4572,2.0432,0.3727,0.1533,0.4800,0.2204,0.1197,...,0.1395,0.2724,0.1317,0.0087,0.0085,0.0087,271,374,0.031283,0.076868
1,Spherical,0.0212,0.0250,2.0046,1.3464,0.3669,0.1426,0.4975,0.2042,0.1080,...,0.1098,0.2560,0.1095,0.0052,0.0061,0.0054,1464,4555,0.021101,0.042300
2,Todd-AO,0.0070,0.0090,4.1510,4.8910,0.4080,0.1720,0.4380,0.2380,0.1330,...,0.1800,0.2270,0.1010,0.0120,0.0040,0.0020,1,3,0.980000,4.067980
3,VistaVision,0.0138,0.0159,2.4273,2.0254,0.3760,0.1495,0.4874,0.2107,0.1177,...,0.1106,0.2480,0.1142,0.0059,0.0057,0.0062,63,83,0.060977,0.148009


In [73]:
fig = px.bar(g, 
             x='process_name', 
             y=['avg_f_per_fr', 'std_f_per_fr'], 
             barmode='group', 
             error_y=g_error['avg_f_per_fr_error'],
             color_discrete_sequence=["#3bbdb6", "#e97bec"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Avg. Faces per Frame by Production Process"))
fig.update_traces(error_y=dict(
                      thickness=2,
                      width=6,
                      color="white"
                  ))
fig.show()

## Distance

In [74]:
g_error = g.merge(cnts,
            how='inner',
            on='process_name')
g_error['avg_dist_error'] = g_error['avg_dist'] * g_error['moe']
g_error

,process_name,avg_size,std_size,avg_f_per_fr,std_f_per_fr,avg_dist,std_dist,gini,avg_disp,std_disp,...,pct_ml,pct_mc,pct_mr,pct_bl,pct_bc,pct_br,count,pop,moe,avg_dist_error
0,CinemaScope,0.0136,0.0147,2.4572,2.0432,0.3727,0.1533,0.4800,0.2204,0.1197,...,0.1395,0.2724,0.1317,0.0087,0.0085,0.0087,271,374,0.031283,0.011659
1,Spherical,0.0212,0.0250,2.0046,1.3464,0.3669,0.1426,0.4975,0.2042,0.1080,...,0.1098,0.2560,0.1095,0.0052,0.0061,0.0054,1464,4555,0.021101,0.007742
2,Todd-AO,0.0070,0.0090,4.1510,4.8910,0.4080,0.1720,0.4380,0.2380,0.1330,...,0.1800,0.2270,0.1010,0.0120,0.0040,0.0020,1,3,0.980000,0.399840
3,VistaVision,0.0138,0.0159,2.4273,2.0254,0.3760,0.1495,0.4874,0.2107,0.1177,...,0.1106,0.2480,0.1142,0.0059,0.0057,0.0062,63,83,0.060977,0.022927


In [75]:
fig = px.bar(g, 
             x='process_name', 
             y=['avg_dist', 'std_dist'],
             error_y=g_error['avg_dist_error'], 
             barmode='group', 
             color_discrete_sequence=["#3bbdb6", "#e97bec"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Distance from Center by Production Process"))
fig.update_traces(error_y=dict(
                      thickness=2,
                      width=6,
                      color="white"
                  ))
fig.show()

## Dispersion

In [76]:
g_error = g.merge(cnts,
            how='inner',
            on='process_name')
g_error['avg_disp_error'] = g_error['avg_disp'] * g_error['moe']
g_error

,process_name,avg_size,std_size,avg_f_per_fr,std_f_per_fr,avg_dist,std_dist,gini,avg_disp,std_disp,...,pct_ml,pct_mc,pct_mr,pct_bl,pct_bc,pct_br,count,pop,moe,avg_disp_error
0,CinemaScope,0.0136,0.0147,2.4572,2.0432,0.3727,0.1533,0.4800,0.2204,0.1197,...,0.1395,0.2724,0.1317,0.0087,0.0085,0.0087,271,374,0.031283,0.006895
1,Spherical,0.0212,0.0250,2.0046,1.3464,0.3669,0.1426,0.4975,0.2042,0.1080,...,0.1098,0.2560,0.1095,0.0052,0.0061,0.0054,1464,4555,0.021101,0.004309
2,Todd-AO,0.0070,0.0090,4.1510,4.8910,0.4080,0.1720,0.4380,0.2380,0.1330,...,0.1800,0.2270,0.1010,0.0120,0.0040,0.0020,1,3,0.980000,0.233240
3,VistaVision,0.0138,0.0159,2.4273,2.0254,0.3760,0.1495,0.4874,0.2107,0.1177,...,0.1106,0.2480,0.1142,0.0059,0.0057,0.0062,63,83,0.060977,0.012848


In [77]:
fig = px.bar(g, 
             x='process_name', 
             y=['avg_disp', 'std_disp'], 
             barmode='group', 
             error_y=g_error['avg_disp_error'], 
             color_discrete_sequence=["#3bbdb6", "#e97bec"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Dispersion by Production Process"))
fig.update_traces(error_y=dict(
                      thickness=2,
                      width=6,
                      color="white"
                  ))
fig.show()

## Gini

In [78]:
g_error = g.merge(cnts,
            how='inner',
            on='process_name')
g_error['gini_error'] = g_error['gini'] * g_error['moe']
g_error

,process_name,avg_size,std_size,avg_f_per_fr,std_f_per_fr,avg_dist,std_dist,gini,avg_disp,std_disp,...,pct_ml,pct_mc,pct_mr,pct_bl,pct_bc,pct_br,count,pop,moe,gini_error
0,CinemaScope,0.0136,0.0147,2.4572,2.0432,0.3727,0.1533,0.4800,0.2204,0.1197,...,0.1395,0.2724,0.1317,0.0087,0.0085,0.0087,271,374,0.031283,0.015016
1,Spherical,0.0212,0.0250,2.0046,1.3464,0.3669,0.1426,0.4975,0.2042,0.1080,...,0.1098,0.2560,0.1095,0.0052,0.0061,0.0054,1464,4555,0.021101,0.010498
2,Todd-AO,0.0070,0.0090,4.1510,4.8910,0.4080,0.1720,0.4380,0.2380,0.1330,...,0.1800,0.2270,0.1010,0.0120,0.0040,0.0020,1,3,0.980000,0.429240
3,VistaVision,0.0138,0.0159,2.4273,2.0254,0.3760,0.1495,0.4874,0.2107,0.1177,...,0.1106,0.2480,0.1142,0.0059,0.0057,0.0062,63,83,0.060977,0.029720


In [79]:
fig = px.bar(g, 
             x='process_name', 
             y=['gini'], 
             barmode='group', 
             error_y=g_error['gini_error'],
             color_discrete_sequence=["#3bbdb6"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Gini Score by Production Process"))
fig.update_traces(error_y=dict(
                      thickness=2,
                      width=6,
                      color="white"
                  ))
fig.show()

## Horizontal Spread

In [80]:
g_error = g.merge(cnts,
            how='inner',
            on='process_name')
g_error['h_spread_error'] = g_error['avg_h_spread'] * g_error['moe']
g_error

,process_name,avg_size,std_size,avg_f_per_fr,std_f_per_fr,avg_dist,std_dist,gini,avg_disp,std_disp,...,pct_ml,pct_mc,pct_mr,pct_bl,pct_bc,pct_br,count,pop,moe,h_spread_error
0,CinemaScope,0.0136,0.0147,2.4572,2.0432,0.3727,0.1533,0.4800,0.2204,0.1197,...,0.1395,0.2724,0.1317,0.0087,0.0085,0.0087,271,374,0.031283,0.004555
1,Spherical,0.0212,0.0250,2.0046,1.3464,0.3669,0.1426,0.4975,0.2042,0.1080,...,0.1098,0.2560,0.1095,0.0052,0.0061,0.0054,1464,4555,0.021101,0.002718
2,Todd-AO,0.0070,0.0090,4.1510,4.8910,0.4080,0.1720,0.4380,0.2380,0.1330,...,0.1800,0.2270,0.1010,0.0120,0.0040,0.0020,1,3,0.980000,0.148960
3,VistaVision,0.0138,0.0159,2.4273,2.0254,0.3760,0.1495,0.4874,0.2107,0.1177,...,0.1106,0.2480,0.1142,0.0059,0.0057,0.0062,63,83,0.060977,0.008396


In [81]:
fig = px.bar(g, 
             x='process_name', 
             y=['avg_h_spread'], 
             barmode='group', 
             error_y=g_error['h_spread_error'],
             color_discrete_sequence=["#3bbdb6"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Horizontal Spread by Production Process"))
fig.update_traces(error_y=dict(
                      thickness=2,
                      width=6,
                      color="white"
                  ))
fig.show()

In [82]:
from scipy import stats

spherical = process_df[process_df['process_name'] == 'Spherical']['avg_h_spread']
cinemascope = process_df[process_df['process_name'] == 'CinemaScope']['avg_h_spread']
vistavision = process_df[process_df['process_name'] == 'VistaVision']['avg_h_spread']

f_stat, p_value = stats.f_oneway(spherical, cinemascope, vistavision)
print(f"F={f_stat:.3f}, p={p_value:.4f}")

F=37.194, p=0.0000


In [83]:
grand_mean = process_df['avg_h_spread'].mean()
ss_between = sum(len(g) * (g.mean() - grand_mean)**2 
                 for g in [spherical, cinemascope, vistavision])
ss_total = sum((process_df['avg_h_spread'] - grand_mean)**2)
eta_squared = ss_between / ss_total
print(f"eta-squared: {eta_squared:.3f}")

eta-squared: 0.040


## Vertical Spread

In [84]:
fig = px.bar(g, x='process_name', y=['avg_v_disc', 'std_v_disc'], barmode='group', color_discrete_sequence=["#3bbdb6", "#e97bec"])
fig.update_layout(layout)
fig.update_layout(title=dict(text="Vertical Spread by Production Process"))
fig.show()

## Grid

In [85]:
grid = g[[
    'process_name',
    'pct_tl',
    'pct_tc',
    'pct_tr',
    'pct_ml',
    'pct_mc',
    'pct_mr',
    'pct_bl',
    'pct_bc',
    'pct_br'
]]
grid

,process_name,pct_tl,pct_tc,pct_tr,pct_ml,pct_mc,pct_mr,pct_bl,pct_bc,pct_br
0,CinemaScope,0.1057,0.2267,0.0982,0.1395,0.2724,0.1317,0.0087,0.0085,0.0087
1,Spherical,0.1142,0.2817,0.1121,0.1098,0.2560,0.1095,0.0052,0.0061,0.0054
2,Todd-AO,0.1080,0.2370,0.1280,0.1800,0.2270,0.1010,0.0120,0.0040,0.0020
3,VistaVision,0.1168,0.2799,0.1126,0.1106,0.2480,0.1142,0.0059,0.0057,0.0062


#### Spherical

In [86]:
spherical = grid[grid['process_name'] == 'Spherical'].drop('process_name', axis=1)
spherical_grid = spherical.values.reshape(3, 3)
spherical_grid_norm = spherical_grid / spherical_grid.sum()
(spherical_grid_norm * 100).round(1)

array([[11.4, 28.2, 11.2],
       [11. , 25.6, 11. ],
       [ 0.5,  0.6,  0.5]])

In [87]:
plot_grid((spherical_grid_norm * 100).round(1), layout=layout, plot_title="Spherical Grid")

#### CinemaScope

In [88]:
scope = grid[grid['process_name'] == 'CinemaScope'].drop('process_name', axis=1)
scope_grid = scope.values.reshape(3, 3)
scope_grid_norm = scope_grid / scope_grid.sum()
(scope_grid_norm * 100).round(1)

array([[10.6, 22.7,  9.8],
       [13.9, 27.2, 13.2],
       [ 0.9,  0.8,  0.9]])

In [89]:
plot_grid((scope_grid_norm * 100).round(1), layout=layout, plot_title="CinemaScope Grid")

#### VistaVision

In [90]:
vista = grid[grid['process_name'] == 'VistaVision'].drop('process_name', axis=1)
vista_grid = vista.values.reshape(3, 3)
vista_grid_norm = vista_grid / vista_grid.sum()
(vista_grid_norm * 100).round(1)

array([[11.7, 28. , 11.3],
       [11.1, 24.8, 11.4],
       [ 0.6,  0.6,  0.6]])

In [91]:
plot_grid((vista_grid_norm * 100).round(1), layout=layout)

#### Todd-AO

In [92]:
todd = grid[grid['process_name'] == 'Todd-AO'].drop('process_name', axis=1)
todd_grid = todd.values.reshape(3, 3)
todd_grid_norm = todd_grid / todd_grid.sum()
(todd_grid_norm * 100).round(1)

array([[10.8, 23.7, 12.8],
       [18. , 22.7, 10.1],
       [ 1.2,  0.4,  0.2]])

In [93]:
plot_grid((todd_grid_norm * 100).round(1), plot_title="Todd-AO Grid", layout=layout)